## Stage 0 — World and LLM Setup

Loads the PR2-apartment world fixture and creates the LLM.

`groundable_type` scopes entity grounding and world serialisation to physical bodies.
It defaults to `Symbol` (all instances) inside `llmr` — passing `Body` gives
a tighter, physical-body-only scope. It is **never** imported inside the library itself.


In [1]:
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, pr2, context = load_pr2_apartment_world()

# Pass Body for physical-body-scoped grounding (tighter than the Symbol default).
groundable_type = Body

print(f"groundable_type : {groundable_type}")
print(f"World loaded.  Scene bodies in SymbolGraph:")
from krrood.symbol_graph.symbol_graph import SymbolGraph
for inst in SymbolGraph().get_instances_of_type(Body):
    from llmr.world.serializer import body_display_name
    print(f"  {body_display_name(inst)}")

INSTRUCTION = "pick up the milk from the table"
print(f"\nInstruction: {INSTRUCTION!r}")


`polytope` failed to import `cvxopt.glpk`.
will use `scipy.optimize.linprog`
Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


groundable_type : <class 'semantic_digital_twin.world_description.world_entity.Body'>
World loaded.  Scene bodies in SymbolGraph:
  base_link
  base_footprint
  base_bellow_link
  base_laser_link
  fl_caster_rotation_link
  fl_caster_l_wheel_link
  fl_caster_r_wheel_link
  fr_caster_rotation_link
  fr_caster_l_wheel_link
  fr_caster_r_wheel_link
  bl_caster_rotation_link
  bl_caster_l_wheel_link
  bl_caster_r_wheel_link
  br_caster_rotation_link
  br_caster_l_wheel_link
  br_caster_r_wheel_link
  torso_lift_link
  l_torso_lift_side_plate_link
  r_torso_lift_side_plate_link
  torso_lift_motor_screw_link
  imu_link
  head_pan_link
  head_tilt_link
  head_plate_frame
  sensor_mount_link
  high_def_frame
  high_def_optical_frame
  double_stereo_link
  wide_stereo_link
  wide_stereo_optical_frame
  wide_stereo_l_stereo_camera_frame
  wide_stereo_l_stereo_camera_optical_frame
  wide_stereo_r_stereo_camera_frame
  wide_stereo_r_stereo_camera_optical_frame
  narrow_stereo_link
  narrow_stereo_

In [2]:
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [3]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")

ROS2 publishers started


In [4]:
# ── LLM ───────────────────────────────────────────────────────────────────────
# Swap provider/model freely — everything downstream accepts any BaseChatModel.
from dotenv import load_dotenv
from llmr.reasoning.llm_config import make_llm, LLMProvider

load_dotenv("../.env")

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini", temperature=0.0)
print(f"LLM ready: {llm.model_name}")

LLM ready: gpt-4o-mini


## Stage 1 — Discover Action Classes

`discover_action_classes()` (from `llmr.pycram_bridge`) scans
`pycram.robot_plans.actions` and returns every concrete class whose name ends
with `"Action"`.  All pycram access is funnelled through `pycram_bridge` —
no direct pycram import elsewhere in llmr.


In [5]:
from llmr.pycram_bridge import discover_action_classes

action_registry = discover_action_classes()

print(f"Discovered {len(action_registry)} action classes:\n")
for name, cls in sorted(action_registry.items()):
    print(f"  {name:30s}  {cls.__module__}.{cls.__qualname__}")


Discovered 24 action classes:

  CarryAction                     pycram.robot_plans.actions.core.robot_body.CarryAction
  CloseAction                     pycram.robot_plans.actions.core.container.CloseAction
  CuttingAction                   pycram.robot_plans.actions.composite.tool_based.CuttingAction
  DetectAction                    pycram.robot_plans.actions.core.misc.DetectAction
  EfficientTransportAction        pycram.robot_plans.actions.composite.transporting.EfficientTransportAction
  FaceAtAction                    pycram.robot_plans.actions.composite.facing.FaceAtAction
  FollowToolCenterPointPathAction  pycram.robot_plans.actions.core.robot_body.FollowToolCenterPointPathAction
  GraspingAction                  pycram.robot_plans.actions.core.pick_up.GraspingAction
  LookAtAction                    pycram.robot_plans.actions.core.navigation.LookAtAction
  MixingAction                    pycram.robot_plans.actions.composite.tool_based.MixingAction
  MoveAndPickUpAction       

## Stage 2 — LLM Classifies Action Type

`classify_action()` is the public classification entry point. Internally it builds a schema-aware action catalog from `PycramIntrospector`, asks the LLM for an `ActionClassification`, and returns the matching Python action class.


In [6]:
from llmr.pycram_bridge import PycramIntrospector

# Preview the action catalog at a high level before calling the public classifier.
# The actual classifier prompt is built inside llmr.reasoning.slot_filler.classify_action().
catalog_introspector = PycramIntrospector()

print("=== Available action classes (schema preview) ===")
for name in sorted(action_registry):
    cls = action_registry[name]
    try:
        schema = catalog_introspector.introspect(cls)
        fields = ", ".join(f"{f.name}:{getattr(f.raw_type, '__name__', f.raw_type)}" for f in schema.fields)
    except Exception:
        fields = "<schema unavailable>"
    print(f"  - {name}: {fields}")

print()
print(f"Instruction: {INSTRUCTION!r}")


=== Available action classes (schema preview) ===
  - CarryAction: arm:Arms, align:bool, tip_link:str, tip_axis:AxisIdentifier, root_link:str, root_axis:AxisIdentifier
  - CloseAction: object_designator:Body, arm:Arms, grasping_prepose_distance:float
  - CuttingAction: object_:Body, tool:SemanticAnnotation, arm:Arms, technique:str, slice_thickness:float
  - DetectAction: technique:DetectionTechnique, state:DetectionState, object_sem_annotation:SemanticAnnotation, region:Region
  - EfficientTransportAction: object_designator:Body, target_location:Pose
  - FaceAtAction: pose:Pose, keep_joint_states:bool
  - FollowToolCenterPointPathAction: target_locations:PoseTrajectory, arm:Arms
  - GraspingAction: object_designator:Body, arm:Arms, grasp_description:GraspDescription
  - LookAtAction: target:Pose, camera:Camera
  - MixingAction: object_:Body, tool:SemanticAnnotation, arm:Arms, technique:str
  - MoveAndPickUpAction: standing_position:Pose, object_designator:Body, arm:Arms, grasp_descript

In [7]:
from llmr.reasoning.slot_filler import classify_action

action_cls = classify_action(
    instruction=INSTRUCTION,
    llm=llm,
    action_registry=action_registry,
)

print("=== LLM classification response ===")
print(f"Resolved class: {action_cls}")
assert action_cls is not None, "LLM could not classify the instruction"
assert action_cls.__name__ == "PickUpAction"


=== LLM classification response ===
Resolved class: <class 'pycram.robot_plans.actions.core.pick_up.PickUpAction'>


## Stage 3 — Build a Fully Underspecified Match

For the notebook we build the `Match` explicitly from the public `PycramIntrospector` schema: every required, prompt-settable action field is set to `...` (Ellipsis), which means the backend must resolve it.

The higher-level `nl_plan()` API does this internally.


In [8]:
from krrood.entity_query_language.query.match import Match
from llmr.pycram_bridge import PycramIntrospector

introspector = PycramIntrospector()
action_schema = introspector.introspect(action_cls)
field_specs = {f.name: f for f in action_schema.fields}

_SKIP_FIELDS = {"id", "plan_node"}
required_fields = [
    f.name
    for f in action_schema.fields
    if not f.is_optional and not f.name.startswith("_") and f.name not in _SKIP_FIELDS
]

print(f"Required prompt-settable fields on {action_cls.__name__}:")
for name in required_fields:
    fspec = field_specs[name]
    type_name = fspec.raw_type.__name__ if isinstance(fspec.raw_type, type) else str(fspec.raw_type)
    print(f"  {name:30s}  kind={fspec.kind.value:10s}  type={type_name}")

#building fully underspecified action instance
match = Match(action_cls)(**{name: ... for name in required_fields})

print()
print(f"Match object : {match}")
print(f"Matches with variables ({len(list(match.matches_with_variables))}):")
for attr in match.matches_with_variables:
    print(f"  field={attr.name_from_variable_access_path:25s}  "
          f"value={attr.assigned_variable._value_!r:10}  "
          f"type={attr.assigned_variable._type_}")


Required prompt-settable fields on PickUpAction:
  object_designator               kind=entity      type=Body
  arm                             kind=enum        type=Arms
  grasp_description               kind=complex     type=GraspDescription

Match object : Match(PickUpAction)
Matches with variables (3):
  field=PickUpAction.object_designator  value=Ellipsis    type=<class 'semantic_digital_twin.world_description.world_entity.Body'>
  field=PickUpAction.arm           value=Ellipsis    type=<enum 'Arms'>
  field=PickUpAction.grasp_description  value=Ellipsis    type=<class 'pycram.datastructures.grasp.GraspDescription'>


In [11]:
required_fields

['object_designator', 'arm', 'grasp_description']

## Stage 3b — Introspect Action Schema with PycramIntrospector

`PycramIntrospector.introspect()` reads the pycram action dataclass and classifies
each field into a `FieldKind`.  This drives the **dynamic LLM prompt**.

| FieldKind | How resolved in `_evaluate` |
|-----------|------------|
| `ENTITY`   | EntityGrounder → Symbol instance (includes Manipulator, Camera) |
| `POSE`     | EntityGrounder → body.global_pose |
| `ENUM`     | string → enum member coercion |
| `COMPLEX`  | lazily introspected; sub-fields reconstructed from dotted SlotValues |
| `PRIMITIVE` | string coerced to bool/int/float/str |

Note: `_evaluate` classifies slots using krrood's **already-resolved** field types
(`attr_match.assigned_variable._type_`) — no full action-class re-introspection.


In [12]:
from llmr.pycram_bridge import FieldKind

# Reuse the schema from Stage 3.
print(f"=== ActionSchema for {action_schema.action_type} ===")
print(f"Docstring: {action_schema.docstring[:120]}...")
print(f"Fields ({len(action_schema.fields)}):")

for fspec in action_schema.fields:
    type_name = fspec.raw_type.__name__ if isinstance(fspec.raw_type, type) else str(fspec.raw_type)
    opt_str = " [optional]" if fspec.is_optional else " [required]"
    print(f"{fspec.name:30s}  kind={fspec.kind.value:10s}  type={type_name}{opt_str}")
    if fspec.docstring:
        print(f"    doc: {fspec.docstring[:100]}")
    if fspec.enum_members:
        print(f"    allowed values: {', '.join(fspec.enum_members)}")
    if fspec.sub_fields:
        print("    sub-fields:")
        for sub in fspec.sub_fields:
            sub_type = sub.raw_type.__name__ if isinstance(sub.raw_type, type) else str(sub.raw_type)
            members_note = f"  allowed: {', '.join(sub.enum_members)}" if sub.enum_members else ""
            print(f"      {sub.name:28s}  kind={sub.kind.value:10s}  type={sub_type}{members_note}")


=== ActionSchema for PickUpAction ===
Docstring: Let the robot pick up an object....
Fields (3):
object_designator               kind=entity      type=Body [required]
    doc: Object designator_description describing the object that should be picked up
arm                             kind=enum        type=Arms [required]
    doc: The arm that should be used for pick up
    allowed values: LEFT, RIGHT, BOTH
grasp_description               kind=complex     type=GraspDescription [required]
    doc: The GraspDescription that should be used for picking up the object
    sub-fields:
      approach_direction            kind=enum        type=ApproachDirection  allowed: FRONT, BACK, RIGHT, LEFT
      vertical_alignment            kind=enum        type=VerticalAlignment  allowed: NoAlignment, TOP, BOTTOM
      manipulator                   kind=entity      type=Manipulator
      rotate_gripper                kind=primitive   type=bool
      manipulation_offset           kind=primitive   type=flo

## Stage 4 — Inspect Free vs Fixed Slots

This is what `LLMBackend._evaluate()` does internally to split the Match expression into free slots (LLM must fill) and fixed slots (already provided by the user).

In [13]:
# Strip 'ClassName.' prefix from name_from_variable_access_path
# (e.g. 'PickUpAction.arm' → 'arm') so downstream lookups work.
_short = lambda n: n.rsplit(".", 1)[-1] if "." in n else n

free_slots  = []   # (field_name, field_type) — LLM must fill these
fixed_slots = {}   # field_name → value     — already known
_full_name_map = {}  # short_name → full access path (for _get_mapped_variable_by_name)

for attr_match in match.matches_with_variables:
    field_name_raw = attr_match.name_from_variable_access_path
    field_name     = _short(field_name_raw)
    value          = attr_match.assigned_variable._value_
    field_type     = attr_match.assigned_variable._type_
    _full_name_map[field_name] = field_name_raw

    if isinstance(value, type(Ellipsis)):
        free_slots.append((field_name, field_type))
    else:
        fixed_slots[field_name] = value

print("FREE slots  (LLM will fill these):")
for name, typ in free_slots:
    print(f"  {name:30s}  type: {typ}")

print("FIXED slots (already known — passed as context to LLM):")
if fixed_slots:
    for name, val in fixed_slots.items():
        print(f"  {name:30s}  =  {val!r}")
else:
    print("  (none — user left everything open)")


FREE slots  (LLM will fill these):
  object_designator               type: <class 'semantic_digital_twin.world_description.world_entity.Body'>
  arm                             type: <enum 'Arms'>
  grasp_description               type: <class 'pycram.datastructures.grasp.GraspDescription'>
FIXED slots (already known — passed as context to LLM):
  (none — user left everything open)


## Stage 4b — All Free Slots go to the LLM

All free slots — including `Manipulator` and `Camera` fields — are sent to the slot filler. They are Symbol subclasses and are grounded from the SymbolGraph like any other entity (`Body`, `Region`, etc.).

There is no longer a separate "context injection" step.


In [14]:
field_specs = {f.name: f for f in action_schema.fields}

# All free slots go to the LLM — Manipulator/Camera are ENTITY, grounded from SymbolGraph
llm_free_slot_names = [name for name, _ in free_slots]

print("LLM-fillable fields (all free slots):")
for name in llm_free_slot_names:
    fspec = field_specs.get(name)
    kind_str = fspec.kind.value if fspec else "unknown"
    print(f"  {name:30s}  kind={kind_str}")


LLM-fillable fields (all free slots):
  object_designator               kind=entity
  arm                             kind=enum
  grasp_description               kind=complex


## Stage 5 — Serialize World via SymbolGraph

`serialize_world_from_symbol_graph(groundable_type)` builds the LLM world-context string from `SymbolGraph` — no world object is passed.

The serializer now emits a structured grounding-oriented snapshot: summary, grounding instructions, scene objects, semantic types, spatial context, and symbol relations. Robot kinematic structures are filtered semantically from robot annotations when available, with the old name-suffix filter only as a fallback.


In [15]:
from llmr.world.serializer import WorldSerializationOptions, serialize_world_from_symbol_graph

world_context = serialize_world_from_symbol_graph(
    groundable_type,
    options=WorldSerializationOptions(
        max_objects=117,
        max_relations=0,
        include_geometry=False,
        include_parent_context=True,
        include_relations=False,
        exclude_robot_structures=True,
    ),
)
print(world_context)

# Expected top-level sections:
#   ## World State Summary
#   ## Grounding Instructions
#   ## Scene Objects
#   ## Available Semantic Types
#   ## Spatial Context
#   ## Symbol Relations


## World State Summary
Objects: 117 visible, Semantic types: 2, Relations: 0 shown

## Grounding Instructions
- Use exact body_name values for entity_description.name when possible.
- Use semantic_type only from Available Semantic Types.
- Use spatial_context for parent, surface, container, or proximity clues.
- Use attributes only for visible distinguishing words such as color, size, or material.

## Scene Objects
| body_name | class | semantic_types | parent_or_surface | xyz | size | notes |
| --- | --- | --- | --- | --- | --- | --- |
| apartment_root | Body | - | map | - | - | - |
| armchair | Body | - | furnitures_root | - | - | - |
| bedside_table | Body | - | furnitures_root | - | - | - |
| breakfast_cereal.stl | Body | Cereal | apartment_root | - | - | - |
| cabinet1 | Body | - | side_A | - | - | - |
| cabinet10 | Body | - | side_B | - | - | - |
| cabinet10_drawer_bottom | Body | - | cabinet10 | - | - | - |
| cabinet10_drawer_middle | Body | - | cabinet10 | - | - | - |
| cabinet

## Stage 6a — Inspect Dynamic Prompt

`run_slot_filler()` builds this prompt internally. This optional inspection cell uses private prompt helpers only so you can see the exact message before the LLM call; normal user code should call `run_slot_filler()`, `resolve_params()`, `resolve_match()`, or `nl_plan()`.


In [16]:
from llmr.reasoning.slot_filler import (
    _build_dynamic_message,
    _select_free_slot_specs,
    _SLOT_FILLER_SYSTEM,
)

prompt_field_specs = _select_free_slot_specs(action_schema, llm_free_slot_names)

user_message = _build_dynamic_message(
    instruction=INSTRUCTION,
    action_cls=action_cls,
    action_schema=action_schema,
    free_slot_names=llm_free_slot_names,
    fixed_slots=fixed_slots,
    world_context=world_context,
    field_specs=prompt_field_specs,
)

print("=== System prompt (shared for all actions) ===")
print(_SLOT_FILLER_SYSTEM)
print()
print("=== Dynamic user message (action-specific) ===")
print(user_message)


=== System prompt (shared for all actions) ===
You are a robot action parameter resolver with strong spatial and physical reasoning.

You receive:
  1. A natural-language instruction from a human operator.
  2. The target robot action type, its description, and all free parameter slots.
  3. Already-fixed slot values (do not change these).
  4. The current world state: scene objects, positions, and semantic annotations.

Your task: for every FREE slot, reason carefully and return a SlotValue.

────────────────────────────────────────────────────
ENTITY SLOTS  (objects / surfaces in the world)
────────────────────────────────────────────────────
Return a SlotValue with:
  - field_name  = the role name exactly as listed
  - entity_description populated:
      name           = noun phrase from the instruction (head noun only, no articles)
      semantic_type  = ontological type from world annotations (null if unknown)
      spatial_context = spatial qualifier from instruction ("on the tab

## Stage 6b — LLM Slot Filling (Core Reasoning)

`run_slot_filler()` sends the dynamic prompt to the LLM and returns an `ActionReasoningOutput` — a structured list of `SlotValue` objects.

For entity slots: each `SlotValue` carries an `EntityDescriptionSchema` (name + semantic_type + spatial_context + attributes) used by `EntityGrounder` in the next stage.

For enum/primitive slots: `SlotValue.value` is the resolved string (e.g. `"LEFT"`, `"TOP"`).

In [17]:
from llmr.reasoning.slot_filler import run_slot_filler
from llmr.schemas.slots import ActionReasoningOutput

output: ActionReasoningOutput = run_slot_filler(
    instruction     = INSTRUCTION,
    action_cls      = action_cls,
    free_slot_names = llm_free_slot_names,
    fixed_slots     = fixed_slots,
    world_context   = world_context,
    llm             = llm,
)

assert output is not None, "Slot filler returned None — check LLM connectivity"

print(f"=== ActionReasoningOutput ===")
print(f"action_type      : {output.action_type}")
print(f"overall_reasoning: {output.overall_reasoning}")
print(f"\nSlots ({len(output.slots)}):")

for sv in output.slots:
    print(f"\n  field_name : {sv.field_name}")
    print(f"  value      : {sv.value!r}")
    if sv.entity_description:
        ed = sv.entity_description
        print(f"  entity_description:")
        print(f"    name            = {ed.name!r}")
        print(f"    semantic_type   = {ed.semantic_type!r}")
        print(f"    spatial_context = {ed.spatial_context!r}")
        print(f"    attributes      = {ed.attributes}")
    print(f"  reasoning  : {sv.reasoning}")

# Expected:
#   field_name : object_designator
#   value      : 'milk_bottle_1'
#   entity_description:
#     name            = 'milk'
#     semantic_type   = 'Milk'
#     spatial_context = 'on the table'
#     attributes      = None
#   reasoning  : 'The milk is identified as a FoodItem on kitchen_table.'
#
#   field_name : arm
#   value      : 'RIGHT'
#   reasoning  : 'Right arm is free; object is on the right side of the robot.'
#
#   field_name : grasp_description.grasp_type
#   value      : 'TOP'
#   reasoning  : 'Approaching from above for a cylindrical bottle.'
#
#   field_name : grasp_description.approach_direction
#   value      : 'FRONT'
#   reasoning  : 'Object is directly in front of the robot.'

=== ActionReasoningOutput ===
action_type      : PickUpAction
overall_reasoning: The robot is instructed to pick up the milk from the table, which is identified as 'milk.stl'. The parameters for the action are set based on typical grasping techniques and the context of the task.

Slots (7):

  field_name : object_designator
  value      : None
  entity_description:
    name            = 'milk.stl'
    semantic_type   = 'Milk'
    spatial_context = 'on the table'
    attributes      = None
  reasoning  : The instruction specifies 'milk', which corresponds to the object 'milk.stl' in the world state. The spatial context indicates it is located on the table.

  field_name : arm
  value      : 'RIGHT'
  reasoning  : The RIGHT arm is chosen for the pick-up action as it is a common choice for right-handed tasks, assuming no specific preference is indicated.

  field_name : grasp_description.approach_direction
  value      : 'FRONT'
  reasoning  : The approach direction is set to FRONT, which

## Stage 7 — Entity Grounding via EntityGrounder

For ENTITY/POSE slots, `LLMBackend` calls `EntityGrounder.ground(entity_description)` which runs a two-tier search in `SymbolGraph`:

- Tier 1: annotation-based lookup from `semantic_type` to SymbolGraph instances
- Tier 2: name-based fallback over instances of `groundable_type`

The LLM sees names and semantic descriptions. Deterministic grounding happens here.


In [18]:
from llmr.world.grounder import EntityGrounder
from llmr.pycram_bridge.introspector import FieldKind
from llmr.schemas.entities import EntityDescriptionSchema

grounder = EntityGrounder(groundable_type)
slot_by_name = {sv.field_name: sv for sv in output.slots}

print("=== Entity grounding results ===")

for fspec in action_schema.fields:
    if fspec.kind not in (FieldKind.ENTITY, FieldKind.POSE, FieldKind.TYPE_REF):
        continue
    sv = slot_by_name.get(fspec.name)
    if sv is None:
        print(f"  {fspec.name}: no SlotValue from LLM")
        continue

    grounding_ed = sv.entity_description or EntityDescriptionSchema(name=sv.value or fspec.name)
    result = grounder.ground(grounding_ed)
    print(f"  {fspec.name}:")
    print(f"    EntityDescription: name={grounding_ed.name!r}  semantic_type={grounding_ed.semantic_type!r}")
    if result.warning:
        print(f"    warning: {result.warning}")
    if result.bodies:
        from llmr.world.serializer import body_display_name
        print(f"    grounded to: {[body_display_name(b) for b in result.bodies]}")
        print(f"    using first: {body_display_name(result.bodies[0])}")
    else:
        print("    grounded to: NOTHING (entity not found in world)")


=== Entity grounding results ===
  object_designator:
    EntityDescription: name='milk.stl'  semantic_type='Milk'
    grounded to: ['milk.stl']
    using first: milk.stl


## Stage 8 — Write Resolved Values into Match → Construct Instance

This cell intentionally mirrors the core resolution loop inside `LLMBackend._evaluate()` so you can inspect the final action instance produced from an already-resolved LLM `ActionReasoningOutput`.

This is a debugging/testing view of the backend internals. For normal usage, use `resolve_params()` in the next stage.


In [19]:
from llmr.backend import (
    _resolve_entity_slot,
    _reconstruct_complex,
    _coerce_enum,
    coerce_primitive,
    _UNRESOLVED,
)
from llmr.pycram_bridge import PycramIntrospector, FieldKind
from llmr.world.grounder import EntityGrounder

# Build a fresh Match so this manual test is repeatable even if earlier cells mutated `match`.
manual_match = Match(action_cls)(**{name: ... for name in required_fields})

_intro = PycramIntrospector()
grounder = EntityGrounder(groundable_type)

print("=== Before assignment — Match variables ===")
for attr in manual_match.matches_with_variables:
    name = _short(attr.name_from_variable_access_path)
    print(f"  {name:30s}  _value_ = {attr.assigned_variable._value_!r}")

slot_by_name = {sv.field_name: sv for sv in output.slots}
resolved_params = {}
manual_full_name_map = {}
manual_free_slots = []

for attr_match in manual_match.matches_with_variables:
    field_name = attr_match.attribute_name
    field_type = attr_match.assigned_variable._type_
    manual_full_name_map[field_name] = attr_match.name_from_variable_access_path
    if isinstance(attr_match.assigned_variable._value_, type(Ellipsis)):
        manual_free_slots.append((field_name, field_type))

print("=== Resolving each free slot using krrood's resolved field types ===")
for field_name, field_type in manual_free_slots:
    # field_type is already resolved by krrood; classify it directly.
    kind = _intro.classify_type(field_type)
    sv = slot_by_name.get(field_name)
    resolved = _UNRESOLVED

    if kind in (FieldKind.ENTITY, FieldKind.POSE, FieldKind.TYPE_REF):
        if sv is not None:
            resolved = _resolve_entity_slot(
                sv,
                grounder,
                kind,
                field_name,
                expected_type=field_type,
            )
        print(f"  {field_name:30s}  {kind.value.upper():8s} -> {resolved}")

    elif kind == FieldKind.ENUM:
        if sv is not None and sv.value:
            resolved = _coerce_enum(sv.value, field_type)
        print(f"  {field_name:30s}  ENUM     -> {resolved!r}")

    elif kind == FieldKind.COMPLEX:
        from llmr.pycram_bridge.introspector import FieldSpec

        sub_schema = _intro.introspect(field_type)
        fspec = FieldSpec(
            name=field_name,
            raw_type=field_type,
            kind=kind,
            sub_fields=sub_schema.fields,
        )
        resolved = _reconstruct_complex(
            field_name=field_name,
            fspec=fspec,
            slot_by_name=slot_by_name,
            grounder=grounder,
            resolved_params=resolved_params,
        )
        print(f"  {field_name:30s}  COMPLEX  -> {resolved!r}")

    else:
        if sv is not None and sv.value is not None:
            resolved = coerce_primitive(sv.value, field_type)
        print(f"  {field_name:30s}  PRIMITIVE -> {resolved!r}")

    if resolved is _UNRESOLVED:
        print(f"    unresolved: leaving {field_name!r} unchanged")
        continue

    resolved_params[field_name] = resolved
    full_name = manual_full_name_map.get(field_name, field_name)
    try:
        manual_match._get_mapped_variable_by_name(full_name)._value_ = resolved
    except Exception as exc:
        print(f"    WARNING: could not write {field_name!r}: {exc}")

manual_match._update_kwargs_from_literal_values()
action_instance = manual_match.construct_instance()

print("=== Constructed action instance from manually resolved Match ===")
print(f"  Type : {type(action_instance).__name__}")
print(f"  Value: {action_instance}")


=== Before assignment — Match variables ===
  object_designator               _value_ = Ellipsis
  arm                             _value_ = Ellipsis
  grasp_description               _value_ = Ellipsis
=== Resolving each free slot using krrood's resolved field types ===
  object_designator               ENTITY   -> Body(name=PrefixedName('None/milk.stl'), id=UUID('f177beb4-1774-4f46-97d8-ffd49c5c1914'), index=219)
  arm                             ENUM     -> RIGHT
  grasp_description               COMPLEX  -> GraspDescription(approach_direction=<ApproachDirection.FRONT: (<AxisIdentifier.X: (1, 0, 0)>, -1)>, vertical_alignment=<VerticalAlignment.TOP: (<AxisIdentifier.Z: (0, 0, 1)>, -1)>, manipulator=ParallelGripper(name=PrefixedName('pr2/right_gripper'), id=UUID('5370ad18-b892-41a1-b0af-822fd8eae345'), root=Body(name=PrefixedName('pr2/r_gripper_palm_link'), id=UUID('37e6c4e8-ebde-4cb0-9870-91adddec4fa5'), index=62), _robot=PR2(neck=Neck(name=PrefixedName('pr2/neck'), id=UUID('e07b8449

In [20]:
print(type(action_instance))
print(action_instance.__dict__.keys())
print(action_instance.object_designator, type(action_instance.object_designator))

<class 'pycram.robot_plans.actions.core.pick_up.PickUpAction'>
dict_keys(['object_designator', 'arm', 'grasp_description'])
Body(name=PrefixedName('None/milk.stl'), id=UUID('f177beb4-1774-4f46-97d8-ffd49c5c1914'), index=219) <class 'semantic_digital_twin.world_description.world_entity.Body'>


## Execution

In [21]:
from pycram.datastructures.enums import Arms
from pycram.locations.locations import CostmapLocation
from pycram.motion_executor import simulated_robot
from pycram.plans.factories import sequential
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.robot_body import ParkArmsAction, MoveTorsoAction
from semantic_digital_twin.datastructures.definitions import TorsoState

# Use the object already resolved by llmr.
object_designator = action_instance.object_designator

# Put ROBOT in a task-ready posture before generating the costmap pose.
setup_plan = sequential(
    [
        ParkArmsAction(Arms.BOTH),
        MoveTorsoAction(TorsoState.HIGH),
    ],
    context=context,
)

with simulated_robot:
    setup_plan.perform()


INFO:base.py::60 perform Performing action ParkArmsAction
INFO:base.py::60 perform Performing action MoveTorsoAction


In [22]:
plan_node = sequential(
    [
        NavigateAction(CostmapLocation(target=action_instance.object_designator.global_pose,
                                       reachable=True,
                                       reachable_arm=action_instance.arm,
                                       grasp_description=action_instance.grasp_description,
                                       context=context).ground(), keep_joint_states=True),
        PickUpAction(
            object_designator=action_instance.object_designator,
            arm=action_instance.arm,
            grasp_description=action_instance.grasp_description,
        ),
    ],
    context=context,
)

print("Navigate + PickUp plan ready:", plan_node)


Navigate + PickUp plan ready: SequentialNode(plan=Plan with 3 nodes, status=<TaskStatus.CREATED: 0>, start_time=datetime.datetime(2026, 4, 15, 15, 41, 1, 655621), end_time=None, reason=None, result=None)


In [23]:
with simulated_robot:
    plan_node.perform()


INFO:base.py::60 perform Performing action NavigateAction
INFO:base.py::60 perform Performing action PickUpAction
INFO:base.py::60 perform Performing action ReachAction


## Stage 9a — Resolve Parameters into an Action Instance

Use the public `resolve_params()` API when you already have a `Match` and want the concrete action instance without creating or executing a `PlanNode`.

Internally this uses `LLMBackend`, grounds entity slots with `EntityGrounder`, reconstructs complex dataclass fields, writes values into the `Match`, and constructs the action instance.


In [ ]:
# Build a fresh Match so this stage is independent of any previous mutations.
params_match = Match(action_cls)(**{name: ... for name in required_fields})

print("Params Match slots:")
for attr in params_match.matches_with_variables:
    print(f"  {attr.name_from_variable_access_path:30s}  {attr.assigned_variable._value_!r}")


In [ ]:
# params_match.__dict__

In [ ]:
from llmr import resolve_params

action_instance = resolve_params(
    match=params_match,
    llm=llm,
    groundable_type=groundable_type,
    instruction=INSTRUCTION,
    strict_required=True,
)

print("=== Constructed action instance ===")
print(f"  Type : {type(action_instance).__name__}")
# print(f"  Value: {action_instance}")


In [ ]:
# action_cls

In [ ]:
print(type(action_instance))
print(action_instance.__dict__.keys())

In [ ]:
print(action_instance.object_designator, type(action_instance.object_designator))
print(action_instance.arm, type(action_instance.arm))
print(action_instance.grasp_description.__dict__.keys(), type(action_instance.grasp_description))
print(action_instance.grasp_description.approach_direction, type(action_instance.grasp_description.approach_direction))
print(action_instance.grasp_description.vertical_alignment, type(action_instance.grasp_description.vertical_alignment))

In [ ]:
# action_instance.grasp_description.manipulator

## Stage 9b— Role 2: `resolve_match()` Plan Node Resolver

`resolve_match()` is the Role 2 entry point when the action type is already known and you want a PyCRAM `PlanNode` for an underspecified `Match`.

Use `resolve_params()` instead when you want only the concrete action instance without plan-node construction.


In [ ]:
from krrood.entity_query_language.query.match import Match
from llmr import resolve_match
from pycram.datastructures.enums import Arms

# Pre-fix arm=RIGHT — LLM fills object_designator + grasp_description only.
power_match = Match(action_cls)(
    object_designator=...,        # free — LLM grounds from world context
    arm=Arms.RIGHT,               # fixed — forced by caller
    grasp_description=...,        # free — LLM reasons about grasp sub-fields
)

print("Match slots:")
for attr in power_match.matches_with_variables:
    name = attr.name_from_variable_access_path
    value = attr.assigned_variable._value_
    status = "FREE (LLM fills)" if isinstance(value, type(Ellipsis)) else f"FIXED = {value!r}"
    print(f"  {name:30s}  {status}")

plan_node = resolve_match(
    match=power_match,
    context=context,
    llm=llm,
    groundable_type=groundable_type,
    instruction=INSTRUCTION,
    strict_required=True,
)

print()
print(f"Plan node  : {plan_node}")
print(f"Backend    : {type(context.query_backend).__name__}")
print(f"groundable : {context.query_backend.groundable_type.__name__}")


## Stage 10 — Simple User API: `nl_plan()` end-to-end

`nl_plan()` runs all previous stages automatically. The user provides only the NL instruction and `groundable_type`. No Match, no action class, no backend — just language.

In [ ]:
from llmr import nl_plan

# groundable_type is optional (defaults to Symbol).
# Pass Body for physical-body-scoped grounding.
plan_node = nl_plan(
    instruction     = INSTRUCTION,
    context         = context,
    llm             = llm,
    groundable_type = groundable_type,
)

print(f"Plan node  : {plan_node}")
print(f"Backend    : {type(context.query_backend).__name__}")
print(f"Instruction: {context.query_backend.instruction!r}")
print(f"groundable : {context.query_backend.groundable_type.__name__}")


## Stage 10b — Navigate Before Pickup

A bare `PickUpAction` only checks whether the object is reachable from the robot's current base pose. If every arm/grasp candidate is unreachable, use PyCRAM's `CostmapLocation` to generate a reachable base pose near the object, then execute `NavigateAction` before `PickUpAction`.

Important: posture setup and final execution must run inside PyCRAM's `simulated_robot` execution environment. If you call `.perform()` without it, PyCRAM logs `Unknown execution type: None`, and the robot state will not be updated for reachability checks.


In [ ]:
from pycram.datastructures.enums import Arms
from pycram.locations.locations import CostmapLocation
from pycram.motion_executor import simulated_robot
from pycram.plans.factories import sequential
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.robot_body import ParkArmsAction, MoveTorsoAction
from semantic_digital_twin.datastructures.definitions import TorsoState

# Use the object already resolved by llmr.
object_designator = action_instance.object_designator

# Put PR2 in a reach-friendly posture before generating the costmap pose.
setup_plan = sequential(
    [
        ParkArmsAction(Arms.BOTH),
        MoveTorsoAction(TorsoState.HIGH),
    ],
    context=context,
)

with simulated_robot:
    setup_plan.perform()


In [ ]:
action_instance.arm

In [ ]:
# pickup_pose = None
# pickup_arm = None
#
# for arm in (Arms.LEFT, Arms.RIGHT):
#     location = CostmapLocation(
#         target=object_designator.global_pose,
#         reachable=True,
#         reachable_arm=arm,
#         context=context,
#     )
#     try:
#         candidate = location.ground()
#     except StopIteration:
#         candidate = None
#
#     if candidate is None:
#         print(f"No reachable navigation pose found for {arm.name}")
#         continue
#
#     pickup_pose = candidate
#     pickup_arm = candidate.arm or arm
#     grasp = pickup_pose.grasp_description
#     print(f"Reachable pickup pose found with {pickup_arm.name}")
#     print("Base pose:", pickup_pose)
#     print("Generated grasp:", grasp.approach_direction, grasp.vertical_alignment)
#     break
#
# assert pickup_pose is not None, "PyCRAM could not find a reachable base pose for pickup"


In [ ]:
# Build a concrete navigate-then-pick plan. This bypasses the LLM for navigation;
# llmr is still used for object grounding, and PyCRAM supplies the feasible base pose/grasp.
plan_node = sequential(
    [
        NavigateAction(CostmapLocation(target=action_instance.object_designator.global_pose,
                                       reachable=True,
                                       reachable_arm=action_instance.arm,
                                       context=context).ground(), keep_joint_states=True),
        PickUpAction(
            object_designator=action_instance.object_designator,
            arm=action_instance.arm,
            grasp_description=action_instance.grasp_description,
        ),
    ],
    context=context,
)

print("Navigate + PickUp plan ready:", plan_node)


## Stage 11 — Execute Navigate + Pickup

This executes the concrete `NavigateAction` followed by `PickUpAction`. Run it inside `simulated_robot`; otherwise PyCRAM will not have an execution backend for motion actions.


In [ ]:
with simulated_robot:
    plan_node.perform()


## Bonus — Multi-Step: `nl_sequential()`

Tests `TaskDecomposer` + `nl_plan()` chained together. Each step gets its own `LLMBackend` with a step-specific instruction and the shared `groundable_type`.

In [ ]:
from llmr.reasoning.decomposer import TaskDecomposer

MULTI_INSTRUCTION = "go to the table, pick up the milk, and put it in the fridge"

decomposer = TaskDecomposer(llm=llm)
decomposed = decomposer.decompose(MULTI_INSTRUCTION)

print("=== Decomposed steps ===")
for i, step in enumerate(decomposed.steps):
    deps = decomposed.dependencies.get(i, [])
    dep_str = f"  (depends on steps {deps})" if deps else ""
    print(f"  [{i}] {step}{dep_str}")

# Expected:
# [0] navigate to the table
# [1] pick up the milk from the table  (depends on steps [0])
# [2] place the milk in the fridge     (depends on steps [1])

In [ ]:
from llmr import nl_sequential

plan_nodes = nl_sequential(
    instruction     = MULTI_INSTRUCTION,
    context         = context,
    llm             = llm,
    groundable_type = groundable_type,
)

print(f"Built {len(plan_nodes)} plan nodes:")
for i, node in enumerate(plan_nodes):
    print(f"  [{i}] {node}")

print("\nExecuting...")
for i, node in enumerate(plan_nodes):
    print(f"\n--- Step {i} ---")
    node.perform()
